# 02_ex_agreement_topic — profile + AI candidate DQ + human validation


In [ ]:
%run 00_env_config
%run 01_data_agreement_template


In [ ]:
from fabricops_kit import (
    get_path,
    lakehouse_table_read,
    profile_dataframe,
    draft_dq_rules,
    run_dq_rule_review_widget,
    get_dq_rule_review_results,
    write_dq_rules,
    load_approved_dq_rules,
    enforce_dq_rules,
    prepare_business_context_profile_input,
    suggest_column_business_contexts,
    extract_column_business_context_suggestions,
    capture_column_business_context,
)
import fabricops_kit.business_context as business_context


In [ ]:
SOURCE_TABLE = "REPLACE_SOURCE_TABLE"
AGREEMENT_ID = "REPLACE_APPROVED_AGREEMENT_ID"
DQ_METADATA_TABLE = "METADATA_DQ_RULES"
df_source = spark.table(SOURCE_TABLE)
profile_rows = profile_dataframe(df_source)
profile_records = [r.asDict(recursive=True) for r in profile_rows.collect()]
metadata_path = get_path(ENV_NAME, "metadata", config=FABRIC_CONFIG)
profile_rows.show(20, truncate=False)


In [ ]:
prepared_context_input = prepare_business_context_profile_input(profile_records, table_name=SOURCE_TABLE, table_context=BUSINESS_CONTEXT)
prepared_context_df = spark.createDataFrame(prepared_context_input)
ai_context_response_df = suggest_column_business_contexts(prepared_profile_df=prepared_context_df, prompt_template=CONFIG.ai_prompt_config.business_context_template)
ai_context_suggestions = extract_column_business_context_suggestions(ai_context_response_df)
capture_column_business_context(ai_context_suggestions, environment_name=ENV_NAME, dataset_name=DATASET_NAME, table_name=SOURCE_TABLE)
# COLUMN_BUSINESS_CONTEXT_FROM_WIDGET is populated asynchronously by widget callbacks.


In [ ]:
dq_candidates = draft_dq_rules(
    profile_df=profile_rows,
    table_name=SOURCE_TABLE,
    business_context=BUSINESS_CONTEXT,
    prompt_template=CONFIG.ai_prompt_config.dq_rule_candidate_template,
)
run_dq_rule_review_widget(
    dq_candidates,
    table_name=SOURCE_TABLE,
)
# After completing widget actions, run this cell again to collect current state.
review_result = get_dq_rule_review_results(
    table_name=SOURCE_TABLE,
    environment_name=ENV_NAME,
    dataset_name=DATASET_NAME,
)
approved_rules = review_result["approved_rules"]
if approved_rules:
    write_dq_rules(
        approved_rules,
        table_name=SOURCE_TABLE,
        metadata_path=metadata_path,
        metadata_table=DQ_METADATA_TABLE,
        action_reason=f"Approved via 02_ex for agreement {AGREEMENT_ID}.",
    )

metadata_dq_rules_df = lakehouse_table_read(metadata_path, DQ_METADATA_TABLE)
approved_rules_for_pipeline = load_approved_dq_rules(metadata_dq_rules_df, table_name=SOURCE_TABLE)
len(approved_rules_for_pipeline)


In [ ]:
dq_result = enforce_dq_rules(
    df_source,
    table_name=SOURCE_TABLE,
    rules=approved_rules_for_pipeline,
)
dq_result.rule_results.show(truncate=False)
dq_result.quarantine_rows.show(20, truncate=False)


In [ ]:
# Optional: persist business-context and governance approvals in their own metadata tables.
# write_column_business_context(spark, business_context.COLUMN_BUSINESS_CONTEXT_FROM_WIDGET, metadata_path, table_name="METADATA_COLUMN_CONTEXT")
# write_column_governance_context(...)
# 03_pc notebooks should load rules from METADATA_DQ_RULES via load_approved_dq_rules(...).
